# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. The workflow demonstrates metadata exploration, data extraction via record set and field `@id`, exploratory data analysis, and visualization.

### Dataset Source
The dataset source is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and inspect the main information about the dataset using `mlcroissant`. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Fields: {hasattr(metadata, 'field')}")
print(f"Record Sets: {hasattr(metadata, 'recordSet')}")


## 2. Data Overview
Review available record sets, their IDs, field IDs, and columns. These `@id` values are needed for all later data extraction.

**Note:** All entity IDs are referenced via their `@id` field per the Croissant specification.


In [ ]:
# Display available record sets and fields with their @id

print("Record Sets (by @id):")
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for record_set in metadata.recordSet:
        print(f"  @id: {record_set['@id']}")
        record_sets.append(record_set['@id'])
        print(f"    name: {record_set.get('name', '-')}")
        print(f"    description: {record_set.get('description', '-')}")
        if 'field' in record_set:
            print(f"    Fields:")
            fields = record_set['field']
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                print(f"      - @id: {field['@id']}   name: {field.get('name', '')}")
                # Get columns inside the field, if any
                if 'column' in field:
                    cols = field['column']
                    if isinstance(cols, dict):
                        cols = [cols]
                    for col in cols:
                        print(f"         column @id: {col['@id']}   name: {col.get('name', '')}")
        print()
else:
    print("No record sets found in metadata. Please check the schema or contact the dataset maintainer.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.


In [ ]:
# Example only: If record_sets non-empty, extract data from all record sets listed by @id

# In this dataset, as of FAIR^2 v1.0 spec, the main recordSet @id is usually 'cr:RecordSet:main' or similar,
# but since this is a FAIR2 dataset, let's dynamically list all available record_sets

dataframes = {}

for rset_id in record_sets:
    # Returns generator of dicts (one per record).
    records = list(dataset.records(record_set=rset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded DataFrame for Record Set @id: {rset_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for Record Set @id: {rset_id}")

# For the rest of the notebook, we can select the first record set for illustration
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"Main Record Set selected for analysis: {main_record_set_id}")
else:
    print("No valid record set found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping. Please adjust `<numeric_field_id>` and `<group_field>` to match the actual `@id` from your dataset.


In [ ]:
# Demonstrate filtering and normalization using a numeric field and grouping.

if len(dataframes) > 0:
    df = dataframes[main_record_set_id]
    print(f"Available fields/columns in DataFrame: {df.columns.tolist()}")

    # Example: Try to select a numeric field and a group field
    # Replace these strings with actual column names (e.g., '@id:cr:Field:peptides' or same)
    # The below picks the first numeric-looking column if available
    numeric_field = None
    for col in df.columns:
        # Simple heuristic: skip typical identifier/string/text columns
        if df[col].dtype in ('int64', 'float64'):
            numeric_field = col
            break
    if not numeric_field:
        # Try to coerce columns to numeric if possible
        for col in df.columns:
            try:
                pd.to_numeric(df[col].dropna())
                numeric_field = col
                break
            except ValueError:
                continue
    print(f"Using numeric field: {numeric_field}")

    # Example group field: pick the first non-numeric column
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break
    print(f"Using group field: {group_field}")

    # Try filtering if a numeric field was found
    if numeric_field:
        # Attempt to coerce to numeric
        df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df_numeric.mean() if df_numeric.notnull().sum() > 0 else 0
        print(f"Using filter threshold: {threshold}")
        filtered_df = df[df_numeric > threshold]
        print(filtered_df.head())

        # Normalization
        if filtered_df.shape[0] > 0:
            filtered_df = filtered_df.assign(**{
                f"{numeric_field}_normalized": (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
            })
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print("No records above threshold for filtering.")
        # Grouping
        if group_field and filtered_df.shape[0] > 0:
            grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by {group_field}:\n", grouped.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframe available for EDA. Ensure record sets contain data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Please update field selection as appropriate for your actual field names and IDs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    df_plot = pd.to_numeric(df[numeric_field], errors='coerce')
    sns.histplot(df_plot.dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        # Boxplot per group
        sns.boxplot(x=df[group_field], y=pd.to_numeric(df[numeric_field], errors='coerce'))
        plt.title(f"{numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No valid DataFrame, numeric field, or group field available for plotting.")

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` with the FAIR^2 dataset package:

* Metadata and record sets can be browsed and loaded by their Croissant `@id`.
* Data extraction allows conversion into pandas DataFrames for flexible analysis.
* Typical EDA operations (filtering, normalization, grouping) and visualization are supported and can be extended.

For complete research workflows or feature engineering, tailor field selection to the specific structure of the dataset and consult the provided Croissant schema (<https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json>).
